# Commits Page

In [ ]:
# annulus/pie chart of number of commits per author for previous full month of public repos tagged
# with feedback-systems

import requests
import datetime
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import os
import calendar
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

# set your GitHub token
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'
GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}

# generate data ranges for the last 6 full months
def get_last_n_month_ranges(n = 6):
    today = datetime.date.today()
    first_day_this_month = datetime.date(today.year, today.month, 1)
    ranges = []
    for i in range(1, n + 1):
        last_month = first_day_this_month - datetime.timedelta(days = 1)
        first_day = datetime.date(last_month.year, last_month.month, 1)
        last_day = datetime.date(last_month.year, last_month.month, calendar.monthrange(last_month.year, last_month.month)[1])
        label = f"{first_day.strftime('%B %Y')}"
        ranges.append((label, first_day.isoformat() + 'T00:00:00Z', last_day.isoformat() + 'T23:59:59Z'))
        first_day_this_month = first_day
    return ranges[::-1]

MONTH_RANGES = get_last_n_month_ranges()

# query GitHub for public [topic]-tagged repos
def search_repos():
    repos = []
    page = 1
    while True:
        url = f'https://api.github.com/search/repositories?q=topic:feedback-systems+is:public&sort=stars&per_page=10&page={page}'
        res = requests.get(url, headers = HEADERS)
        res.raise_for_status()
        data = res.json()
        repos.extend(data['items'])
        if 'next' not in res.links:
            break
        page += 1
    return [(repo['owner']['login'], repo['name']) for repo in repos]

# fetch commits for a repo between date range
def fetch_commits(owner, repo, since, until):
    commits = []
    page = 1
    while True:
        url = f'https://api.github.com/repos/{owner}/{repo}/commits'
        params = {
            'since': since,
            'until': until,
            'per_page': 100,
            'page': page
        }
        res = requests.get(url, headers = HEADERS, params = params)
        if res.status_code == 409: # empty repo
            break
        res.raise_for_status()
        data = res.json()
        if not data:
            break
        commits.extend(data)
        page += 1
    return commits

# aggregate commits by author for a given month
def aggregate_authors_for_month(since, until):
    author_counts = Counter()
    for owner, repo in search_repos():
        commits = fetch_commits(owner, repo, since, until)
        for commit in commits:
            author = commit.get('commit', {}).get('author', {}).get('name', 'Unknown')
            author_counts[author] += 1
    return author_counts

# plot pie chart
def plot_pie(author_counts, month_label):
    if not author_counts:
        print('No commits found')
        return

    top_10 = author_counts.most_common(10)
    others_sum = sum(count for _, count in author_counts.items() if (name := _) not in dict(top_10))

    labels = [name for name, _ in top_10] + ['Others']
    sizes = [count for _, count in top_10] + [others_sum]

    fig, ax = plt.subplots(figsize = (8, 8))
    wedges, texts, autotexts = ax.pie(
        sizes, labels = None, autopct = '%1.1f%%', startangle = 140, wedgeprops = dict(width = 0.4)
    )
    ax.legend(wedges, labels, title = 'Authors', loc = 'center left', bbox_to_anchor = (1, 0, 0.5, 1))
    ax.set_title(f'Commits per Author (Topic: feedback-systems, {month_label})')
    ax.axis('equal')
    plt.tight_layout()
    plt.show()

# interactive multi-month selection
options = [label for label, _, _ in MONTH_RANGES]
month_selector = widgets.SelectMultiple(
    options = options,
    value = [options[-1]],
    description = 'Months:',
    disabled = False
)

run_button = widgets.Button(description = 'Generate Chart')

output = widgets.Output()

def on_button_click(b):
    output.clear_output()
    with output:
        combined_counts = Counter()
        for label, since, until in MONTH_RANGES:
            if label in month_selector.value:
                print(f'Fetching commits for {label}...')
                counts = aggregate_authors_for_month(since, until)
                combined_counts.update(counts)
        plot_pie(combined_counts, ', '.join(month_selector.value))

run_button.on_click(on_button_click)

display(widgets.VBox([month_selector, run_button, output]))

In [ ]:
# graphql-based table: Contributor | Sum of Additions | Sum of Deletions

import requests
import datetime
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import os
import calendar
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

GRAPHQL_URL = 'https://api.github.com/graphql'

# set your GitHub token
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'
if 'GITHUB_TOKEN' not in os.environ:
    raise EnvironmentError("GITHUB_TOKEN not found. Please set it with your GitHub personal access token.")

GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}

# generate last 6 full month ranges
def get_last_n_month_ranges(n = 6):
  today = datetime.date.today()
  first_day_this_month = datetime.date(today.year, today.month, 1)
  ranges = []
  for _ in range(n):
    last_day_last_month = first_day_this_month - datetime.timedelta(days = 1)
    first_day_last_month = datetime.date(last_day_last_month.year, last_day_last_month.month, 1)
    label = f'{first_day_last_month.strftime("%B %Y")}'
    since = first_day_last_month.isoformat()
    until = last_day_last_month.isoformat()
    ranges.append((label, since, until))
    first_day_this_month = first_day_last_month
  return ranges[::-1]

MONTH_RANGES = get_last_n_month_ranges()

# get repos tagged with topic
def get_repos_by_topic(topic = 'python', min_stars = 79000):
  repos = []
  page = 1
  while True:
    url = f'https://api.github.com/search/repositories?q=topic:{topic}+stars:>={min_stars}+is:public&sort=stars&order=desc&per_page=10&page={page}'
    res = requests.get(url, headers = HEADERS)

    # rate limit or invalid token check
    if res.status_code == 403:
      raise RuntimeError(f'403 Forbidden - Likely rate limited or invalid token: {res.json().get("message")}')

    res.raise_for_status()
    data = res.json()
    repos.extend([(repo['owner']['login'], repo['name']) for repo in data['items']])
    if 'next' not in res.links:
      break
    page += 1
  return repos

# graphql query per repo
QUERY_TEMPLATE = '''
query($owner:String!, $name:String!, $since:DateTime!, $until:DateTime!) {
  repository(owner:$owner, name:$name) {
    pullRequests(states:MERGED, first:100, orderBy:{field:UPDATED_AT, direction:DESC}) {
      nodes {
        mergedAt
        additions
        deletions
        author {
          login
        }
      }
    }
  }
}
'''

def run_query(owner, name, since, until):
  variables = {
      'owner': owner,
      'name': name,
      'since': since + 'T00:00:00Z',
      'until': until + 'T23:59:59Z'
  }
  res = requests.post(
      GRAPHQL_URL,
      json = {'query': QUERY_TEMPLATE, 'variables': variables},
      headers = HEADERS
  )
  res.raise_for_status()
  return res.json()

# aggregate data
def collect_contributions(since, until):
  results = defaultdict(lambda: {'additions': 0, 'deletions': 0})
  repos = get_repos_by_topic()

  for owner, name in repos:
    print(f'Fetching: {owner}/{name}')
    try:
      data = run_query(owner, name, since, until)
      prs = data.get('data', {}).get('repository', {}).get('pullRequests', {}).get('nodes', [])
      for pr in prs:
        merged = pr.get('mergedAt')
        if not merged or not since <= merged[:10] <= until:
          continue
        author = pr.get('author', {}).get('login') or 'Unknown'
        results[author]['additions'] += pr.get('additions', 0)
        results[author]['deletions'] += pr.get('deletions', 0)
    except Exception as e:
      print(f'Skipping {owner}/{name} due to error: {e}')
      continue

  return results

# build dataframe
def build_dataframe(since, until):
  data = collect_contributions(since, until)
  rows = [
      {
          'Contributor': author,
          'Sum of Additions': counts['additions'],
          'Sum of Deletions': counts['deletions']
      }
      for author, counts in data.items()
  ]
  df = pd.DataFrame(rows)
  # Check if 'Sum of Additions' column exists before sorting
  if 'Sum of Additions' in df.columns:
    df = df.sort_values(by = 'Sum of Additions', ascending = False).reset_index(drop = True)
  return df

# UI widget
options = [label for label, _, _ in MONTH_RANGES]
selector = widgets.Dropdown(options = options, description = 'Month:')
run_button = widgets.Button(description = 'Generate Table')
output = widgets.Output()

def on_click_run(b):
  output.clear_output()
  with output:
    for label, since, until in MONTH_RANGES:
      if label == selector.value:
        print(f'Loading data for {label}...')
        try:
          df = build_dataframe(since, until)
          if df.empty:
            print('No data found for this month.')
          else:
            display(df)
        except Exception as e:
          print(f'Error fetching data: {e}')
        break

run_button.on_click(on_click_run)
display(widgets.VBox([selector, run_button, output]))

In [ ]:
# GitHub PR Merge Time Bar Chart

import requests
import datetime
import os
import calendar
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from collections import defaultdict
from IPython.display import display

GRAPHQL_URL = 'https://api.github.com/graphql'

# --- GitHub Auth ---
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'
if 'GITHUB_TOKEN' not in os.environ:
    raise EnvironmentError("GITHUB_TOKEN not found. Please set it.")

GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}

# --- Last 6 full month ranges ---
def get_last_n_month_ranges(n=6):
    today = datetime.date.today()
    first_day_this_month = datetime.date(today.year, today.month, 1)
    ranges = []
    for _ in range(n):
        last_day_last_month = first_day_this_month - datetime.timedelta(days=1)
        first_day_last_month = datetime.date(last_day_last_month.year, last_day_last_month.month, 1)
        label = f'{first_day_last_month.strftime("%B %Y")}'
        since = first_day_last_month.isoformat()
        until = last_day_last_month.isoformat()
        ranges.append((label, since, until))
        first_day_this_month = first_day_last_month
    return ranges[::-1]  # latest first

MONTH_RANGES = get_last_n_month_ranges()

# --- Get repos with topic and min stars ---
def get_repos(topic='javascript', min_stars=100000):
    repos = []
    page = 1
    while True:
        url = f'https://api.github.com/search/repositories?q=topic:{topic}+stars:>={min_stars}+is:public&sort=stars&per_page=10&page={page}'
        res = requests.get(url, headers=HEADERS)
        if res.status_code == 403:
            raise RuntimeError(f'Rate limit exceeded or bad token: {res.json().get("message")}')
        res.raise_for_status()
        data = res.json()
        repos.extend([(repo['owner']['login'], repo['name']) for repo in data['items']])
        if 'next' not in res.links:
            break
        page += 1
    return repos

# --- GraphQL PR query ---
QUERY_TEMPLATE = '''
query($owner:String!, $name:String!) {
  repository(owner:$owner, name:$name) {
    pullRequests(states:MERGED, first:100, orderBy:{field:UPDATED_AT, direction:DESC}) {
      nodes {
        createdAt
        mergedAt
      }
    }
  }
}
'''

# --- Run GraphQL query ---
def run_query(owner, name):
    res = requests.post(
        GRAPHQL_URL,
        json={"query": QUERY_TEMPLATE, "variables": {"owner": owner, "name": name}},
        headers=HEADERS
    )
    res.raise_for_status()
    return res.json()

# --- Compute average PR merge time (in days) ---
def compute_avg_merge_times(since, until):
    repos = get_repos()
    results = {}

    for owner, name in repos:
        print(f'Fetching: {owner}/{name}')
        try:
            data = run_query(owner, name)
            prs = data.get('data', {}).get('repository', {}).get('pullRequests', {}).get('nodes', [])

            durations = []
            for pr in prs:
                created = pr.get('createdAt')
                merged = pr.get('mergedAt')
                if not created or not merged:
                    continue
                if not (since <= merged[:10] <= until):
                    continue

                t1 = datetime.datetime.fromisoformat(created.replace('Z', '+00:00'))
                t2 = datetime.datetime.fromisoformat(merged.replace('Z', '+00:00'))
                delta = t2 - t1
                durations.append(delta.total_seconds() / 86400)  # seconds to days

            if durations:
                avg_days = round(sum(durations) / len(durations), 1)
                results[f'{owner}/{name}'] = avg_days
        except Exception as e:
            print(f'Skipped {owner}/{name} due to error: {e}')
            continue

    return results

# --- Plot bar chart ---
def plot_bar_chart(avg_times, month_label):
    if not avg_times:
        print("No data to display.")
        return

    df = pd.DataFrame(list(avg_times.items()), columns=['Repo', 'Avg Merge Time (days)'])
    df = df.sort_values(by='Avg Merge Time (days)').reset_index(drop=True)

    plt.figure(figsize=(10, 6))
    plt.barh(df['Repo'], df['Avg Merge Time (days)'])
    plt.xlabel('Avg Merge Time (days)')
    plt.title(f'Average PR Merge Time per Repo for JavaScript topic with 100k+ stars ({month_label})')
    plt.tight_layout()
    plt.show()

# --- Widgets UI ---
options = [label for label, _, _ in MONTH_RANGES]
selector = widgets.Dropdown(options=options[::-1], value=options[-1], description='Month:')
run_button = widgets.Button(description='Generate Chart')
output = widgets.Output()

def on_click_run(b):
    output.clear_output()
    with output:
        for label, since, until in MONTH_RANGES:
            if label == selector.value:
                print(f'Generating chart for {label}...')
                data = compute_avg_merge_times(since, until)
                plot_bar_chart(data, label)
                break

run_button.on_click(on_click_run)
display(widgets.VBox([selector, run_button, output]))

In [ ]:
import requests
import datetime
import pandas as pd
from collections import defaultdict
import os
import ipywidgets as widgets
from IPython.display import display, HTML

GRAPHQL_URL = 'https://api.github.com/graphql'

# --- GitHub Auth ---
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'
if 'GITHUB_TOKEN' not in os.environ:
    raise EnvironmentError("GITHUB_TOKEN not found. Please set it.")

GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}

# Generate last 6 full month ranges
def get_last_n_month_ranges(n=6):
    today = datetime.date.today()
    first_day_this_month = datetime.date(today.year, today.month, 1)
    ranges = []
    for _ in range(n):
        last_day = first_day_this_month - datetime.timedelta(days=1)
        first_day = datetime.date(last_day.year, last_day.month, 1)
        label = first_day.strftime("%B %Y")
        since = first_day.isoformat()
        until = last_day.isoformat()
        ranges.append((label, since, until))
        first_day_this_month = first_day
    return ranges[::-1]

MONTH_RANGES = get_last_n_month_ranges()

# Repos with topic:javascript and stars>=100000
def get_repos(topic = 'javascript', stars = 100000):
    repos = []
    url = f'https://api.github.com/search/repositories?q=topic:{topic}+stars:>={str(stars)}+is:public&sort=stars&order=desc&per_page=10'
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    data = res.json()
    repos.extend([(repo['owner']['login'], repo['name']) for repo in data['items']])
    return repos

# GraphQL query to get commits
def get_commit_count(owner, name, since, until):
    query = '''
    query($owner: String!, $name: String!, $since: GitTimestamp!, $until: GitTimestamp!) {
      repository(owner: $owner, name: $name) {
        defaultBranchRef {
          target {
            ... on Commit {
              history(since: $since, until: $until) {
                totalCount
              }
            }
          }
        }
      }
    }
    '''
    variables = {
        "owner": owner,
        "name": name,
        "since": since + "T00:00:00Z",
        "until": until + "T23:59:59Z"
    }
    res = requests.post(GRAPHQL_URL, json={'query': query, 'variables': variables}, headers=HEADERS)
    res.raise_for_status()
    data = res.json()
    count = data.get('data', {}).get('repository', {}).get('defaultBranchRef', {}).get('target', {}).get('history', {}).get('totalCount', 0)
    return count

# Aggregate commit counts per month
def fetch_month_commit_count(since, until):
    repos = get_repos()
    total_commits = 0
    for owner, name in repos:
        try:
            count = get_commit_count(owner, name, since, until)
            total_commits += count
        except Exception as e:
            print(f"Skipping {owner}/{name}: {e}")
    return total_commits

# Render KPI widget
def display_kpi_widget():
    options = [label for label, _, _ in MONTH_RANGES]
    selector = widgets.Dropdown(options=options[::-1], value=options[-1], description='Month:')
    output = widgets.Output()

    def render_kpi(label):
        idx = next(i for i, (l, _, _) in enumerate(MONTH_RANGES) if l == label)
        label, since, until = MONTH_RANGES[idx]
        prev_label, prev_since, prev_until = MONTH_RANGES[idx - 1] if idx > 0 else (None, None, None)

        current_count = fetch_month_commit_count(since, until)
        prev_count = fetch_month_commit_count(prev_since, prev_until) if prev_label else 0

        if prev_count > 0:
            change = ((current_count - prev_count) / prev_count) * 100
        else:
            change = 0.0

        arrow = '▲' if change > 0 else '▼'
        color = 'green' if change > 0 else 'red'

        html = f'''
        <div style="border:1px solid #ccc; padding:20px; border-radius:10px; width:300px;">
          <h4 style="margin:0; font-weight:normal;">Number of contributor commits in {label}</h4>
          <h1 style="margin:10px 0;">{current_count:,}</h1>
          <p style="color:{color}; font-size:16px; margin:0;">{arrow} {abs(change):.1f}% <span style='font-size:12px; color:black;'>MoM</span></p>
        </div>
        '''
        display(HTML(html))

    def on_change(change):
        output.clear_output()
        with output:
            render_kpi(change.new)

    selector.observe(on_change, names='value')

    with output:
        render_kpi(selector.value)

    display(widgets.VBox([selector, output]))

# Run
display_kpi_widget()

In [ ]:
# Modified KPI box with YoY % change and arrow
import requests
import datetime
import os
from collections import defaultdict
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

# --- GitHub Auth ---
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'
if 'GITHUB_TOKEN' not in os.environ:
    raise EnvironmentError("GITHUB_TOKEN not found. Please set it.")

GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}
GRAPHQL_URL = 'https://api.github.com/graphql'

# Last 6 months (excluding current)
def get_last_6_months():
    today = datetime.date.today()
    first_this_month = datetime.date(today.year, today.month, 1)
    months = []
    for i in range(1, 7):
        month = first_this_month - datetime.timedelta(days=1)
        first_day = datetime.date(month.year, month.month, 1)
        if month.month == 12:
            next_month = datetime.date(month.year + 1, 1, 1)
        else:
            next_month = datetime.date(month.year, month.month + 1, 1)
        last_day = next_month - datetime.timedelta(days=1)
        months.append((first_day.strftime('%B %Y'), first_day.isoformat(), last_day.isoformat()))
        first_this_month = first_day
    return months[::-1]

MONTH_RANGES = get_last_6_months()

# Get repos
def get_top_repos(topic='javascript', stars=100000):
    repos = []
    url = f'https://api.github.com/search/repositories?q=topic:{topic}+stars:>={str(stars)}+is:public&per_page=5'
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    items = res.json()['items']
    for repo in items:
        repos.append((repo['owner']['login'], repo['name']))
    return repos

# GraphQL query
QUERY = '''
query($owner:String!, $name:String!, $since:GitTimestamp!, $until:GitTimestamp!) {
  repository(owner: $owner, name: $name) {
    defaultBranchRef {
      target {
        ... on Commit {
          history(since: $since, until: $until) {
            totalCount
          }
        }
      }
    }
  }
}'''

def get_commit_count(owner, repo, since, until):
    variables = {'owner': owner, 'name': repo, 'since': since + 'T00:00:00Z', 'until': until + 'T23:59:59Z'}
    res = requests.post(GRAPHQL_URL, json={'query': QUERY, 'variables': variables}, headers=HEADERS)
    res.raise_for_status()
    data = res.json()
    try:
        return data['data']['repository']['defaultBranchRef']['target']['history']['totalCount']
    except:
        return 0

# 12mo commit count before given month
def get_12mo_commits(selected_since):
    since_date = datetime.date.fromisoformat(selected_since)
    total = 0
    repos = get_top_repos()
    for _ in range(12):
        month = since_date - datetime.timedelta(days=1)
        first_day = datetime.date(month.year, month.month, 1)
        if month.month == 12:
            next_month = datetime.date(month.year + 1, 1, 1)
        else:
            next_month = datetime.date(month.year, month.month + 1, 1)
        last_day = next_month - datetime.timedelta(days=1)
        for owner, name in repos:
            total += get_commit_count(owner, name, first_day.isoformat(), last_day.isoformat())
        since_date = first_day
    return total

# KPI rendering
def render_kpi_box(value, prev_value, month_label):
    # Percent change
    if prev_value == 0:
        pct_change = 0
    else:
        pct_change = ((value - prev_value) / prev_value) * 100
    arrow = '▲' if pct_change >= 0 else '▼'
    color = 'green' if pct_change >= 0 else 'red'
    pct_string = f"<span style='color:{color};font-size:16px;'>{arrow} {abs(pct_change):.1f}%</span>"

    html = f'''
    <div style="border:1px solid #ccc;padding:20px;border-radius:10px;width:300px;background:#f9f9f9">
        <h4 style="margin:0 0 10px 0">Total commits in past 12 months<br><small style='color:#888;'>Excludes {month_label}</small></h4>
        <div style="font-size:32px;font-weight:bold;">{value}</div>
        <div style="margin-top:5px;font-size:14px;color:#666">YoY: {pct_string}</div>
    </div>'''
    display(HTML(html))

# UI
month_selector = widgets.Dropdown(options=[label for label, _, _ in MONTH_RANGES], value=MONTH_RANGES[-1][0])
output = widgets.Output()

def on_month_change(change):
    output.clear_output()
    with output:
        for i, (label, since, until) in enumerate(MONTH_RANGES):
            if label == month_selector.value:
                print(f"Fetching data for: {label} (12mo before)")
                current_12mo = get_12mo_commits(since)
                # compute start of previous 12mo window (12 months earlier)
                since_date = datetime.date.fromisoformat(since)
                since_1y_ago = (since_date - datetime.timedelta(days=365)).isoformat()
                previous_12mo = get_12mo_commits(since_1y_ago)
                render_kpi_box(current_12mo, previous_12mo, label)
                break

month_selector.observe(on_month_change, names='value')
display(widgets.VBox([month_selector, output]))
on_month_change(None)

In [ ]:
import requests
import datetime
import os
from collections import defaultdict
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

# --- GitHub Auth ---
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'
if 'GITHUB_TOKEN' not in os.environ:
    raise EnvironmentError("GITHUB_TOKEN not found. Please set it.")

GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}
GRAPHQL_URL = 'https://api.github.com/graphql'

# --- Month Selector Utility ---
def get_last_6_months():
    today = datetime.date.today()
    first_this_month = datetime.date(today.year, today.month, 1)
    months = []
    for i in range(1, 7):
        last_month = first_this_month - datetime.timedelta(days=1)
        first_day = datetime.date(last_month.year, last_month.month, 1)
        last_day = datetime.date(last_month.year, last_month.month, (datetime.date(last_month.year, last_month.month + 1, 1) - datetime.timedelta(days=1)).day)
        months.append((first_day.strftime('%B %Y'), first_day.isoformat(), last_day.isoformat()))
        first_this_month = first_day
    return months[::-1]

MONTH_RANGES = get_last_6_months()

# --- Repo Fetch ---
def get_top_repos(topic='explanatory-math-videos', stars=0):
    repos = []
    url = f'https://api.github.com/search/repositories?q=topic:{topic}+stars:>={stars}+is:public&per_page=5'
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    for repo in res.json().get('items', []):
        repos.append((repo['owner']['login'], repo['name']))
    return repos

# --- GraphQL PR History Query ---
QUERY = '''
query($owner:String!, $name:String!, $since:GitTimestamp!, $until:GitTimestamp!) {
  repository(owner: $owner, name: $name) {
    pullRequests(first: 100, states: MERGED, orderBy: {field: CREATED_AT, direction: ASC}) {
      nodes {
        createdAt
        author {
          login
        }
      }
    }
  }
}
'''

def get_first_time_commits(owner, repo, since, until):
    variables = {
        'owner': owner,
        'name': repo,
        'since': since + 'T00:00:00Z',
        'until': until + 'T23:59:59Z'
    }
    res = requests.post(GRAPHQL_URL, json={'query': QUERY, 'variables': variables}, headers=HEADERS)
    res.raise_for_status()
    prs = res.json().get('data', {}).get('repository', {}).get('pullRequests', {}).get('nodes', [])

    seen_authors = set()
    count = 0
    for pr in prs:
        author = pr.get('author', {}).get('login')
        created_at = pr.get('createdAt')[:10] if pr.get('createdAt') else None
        if not author or not created_at:
            continue
        if created_at >= since and created_at <= until and author not in seen_authors:
            seen_authors.add(author)
            count += 1
    return count

# --- Aggregate PR Counts ---
def get_first_time_commit_total(since, until):
    repos = get_top_repos()
    total = 0
    for owner, name in repos:
        total += get_first_time_commits(owner, name, since, until)
    return total

# --- KPI Box with MoM % change ---
def render_kpi_box(current, previous, month_label):
    if previous == 0:
        percent = 'N/A'
        arrow = ''
        color = '#888'
    else:
        diff = current - previous
        change = (diff / previous) * 100
        arrow = '▲' if change >= 0 else '▼'
        color = 'green' if change >= 0 else 'red'
        percent = f"{abs(change):.1f}%"

    html = f'''
    <div style="border:1px solid #ccc;padding:20px;border-radius:10px;width:350px;background:#f9f9f9">
        <h4 style="margin:0 0 10px 0">Commits from first-time contributors in <b>{month_label}</b></h4>
        <div style="font-size:32px;font-weight:bold;">{current}</div>
        <div style="font-size:16px;color:#888;margin-top:5px">MoM <span style='color:{color};font-weight:bold'>{arrow} {percent}</span></div>
    </div>'''
    display(HTML(html))

# --- UI Widget ---
month_selector = widgets.Dropdown(options=[label for label, _, _ in MONTH_RANGES], value=MONTH_RANGES[-1][0])
output = widgets.Output()

def on_month_change(change):
    output.clear_output()
    with output:
        for idx, (label, since, until) in enumerate(MONTH_RANGES):
            if label == month_selector.value:
                print(f"Loading data for: {label}...")
                current = get_first_time_commit_total(since, until)
                if idx > 0:
                    _, prev_since, prev_until = MONTH_RANGES[idx - 1]
                    previous = get_first_time_commit_total(prev_since, prev_until)
                else:
                    previous = 0
                render_kpi_box(current, previous, label)
                break

month_selector.observe(on_month_change, names='value')
display(widgets.VBox([month_selector, output]))
on_month_change(None)

In [ ]:
import requests
import datetime
import os
from collections import defaultdict
import ipywidgets as widgets
from IPython.display import display, HTML

# --- GitHub Auth ---
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'
if 'GITHUB_TOKEN' not in os.environ:
    raise EnvironmentError("GITHUB_TOKEN not found. Please set it.")

GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}
GRAPHQL_URL = 'https://api.github.com/graphql'

# Generate last 6 full month ranges (excluding current month)
def get_last_6_months():
    today = datetime.date.today()
    first_this_month = datetime.date(today.year, today.month, 1)
    months = []
    for i in range(1, 7):
        month = first_this_month - datetime.timedelta(days=1)
        first_day = datetime.date(month.year, month.month, 1)
        last_day = datetime.date(month.year, month.month, (datetime.date(month.year, month.month + 1, 1) - datetime.timedelta(days=1)).day)
        months.append((first_day.strftime('%B %Y'), first_day.isoformat(), last_day.isoformat()))
        first_this_month = first_day
    return months[::-1]

MONTH_RANGES = get_last_6_months()

# Get top repos

def get_top_repos(topic = 'explanatory-math-videos', stars = 0):
    repos = []
    url = f'https://api.github.com/search/repositories?q=topic:{topic}+stars:>={str(stars)}+is:public&per_page=5'
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    for item in res.json()['items']:
        repos.append((item['owner']['login'], item['name']))
    return repos

# GraphQL query for commits
QUERY = '''
query($owner:String!, $name:String!, $since:GitTimestamp!, $until:GitTimestamp!) {
  repository(owner:$owner, name:$name) {
    defaultBranchRef {
      target {
        ... on Commit {
          history(since:$since, until:$until, first:100) {
            edges {
              node {
                committedDate
                author {
                  user {
                    login
                  }
                }
              }
            }
          }
        }
      }
    }
  }
}
'''

def get_commit_authors(owner, name, since, until):
    variables = {
        'owner': owner,
        'name': name,
        'since': since + 'T00:00:00Z',
        'until': until + 'T23:59:59Z'
    }
    res = requests.post(GRAPHQL_URL, json={'query': QUERY, 'variables': variables}, headers=HEADERS)
    res.raise_for_status()
    data = res.json()
    try:
        edges = data['data']['repository']['defaultBranchRef']['target']['history']['edges']
        return [e['node']['author']['user']['login'] for e in edges if e['node']['author']['user'] is not None]
    except:
        return []

# Aggregate repeat contributors
def get_repeat_commit_count(since, until):
    repos = get_top_repos()
    repeat_total = 0
    for owner, name in repos:
        contributors = set()
        all_previous = set()

        # Get contributors before the selected month
        past_since = '2000-01-01'
        past_until = (datetime.date.fromisoformat(since) - datetime.timedelta(days=1)).isoformat()
        previous_authors = get_commit_authors(owner, name, past_since, past_until)
        all_previous.update(previous_authors)

        # Get contributors in current selected month
        current_authors = get_commit_authors(owner, name, since, until)
        for login in current_authors:
            if login in all_previous:
                repeat_total += 1
    return repeat_total

# KPI box render
def render_kpi_box(value, percent_change, month_label):
    color = 'green' if percent_change > 0 else 'red'
    arrow = '&#9650;' if percent_change > 0 else '&#9660;'
    html = f'''
    <div style="border:1px solid #ccc;padding:20px;border-radius:10px;width:300px;background:#f9f9f9">
        <h4 style="margin:0 0 10px 0">Repeat contributor commits in {month_label}</h4>
        <div style="font-size:32px;font-weight:bold;">{value}</div>
        <div style="font-size:16px;color:#888;">MoM &nbsp; <span style='color:{color};font-weight:bold;'>{arrow} {abs(percent_change)}%</span></div>
    </div>'''
    display(HTML(html))

# UI
month_selector = widgets.Dropdown(options=[label for label, _, _ in MONTH_RANGES], value=MONTH_RANGES[-1][0])
output = widgets.Output()

def on_month_change(change):
    output.clear_output()
    with output:
        idx = next(i for i, (label, _, _) in enumerate(MONTH_RANGES) if label == month_selector.value)
        if idx >= 1:
            curr_label, curr_since, curr_until = MONTH_RANGES[idx]
            prev_label, prev_since, prev_until = MONTH_RANGES[idx - 1]
            print(f"Fetching repeat commits for: {curr_label} vs {prev_label}")
            current = get_repeat_commit_count(curr_since, curr_until)
            previous = get_repeat_commit_count(prev_since, prev_until)
            change = 0
            if previous != 0:
                change = round((current - previous) / previous * 100)
            render_kpi_box(current, change, curr_label)

month_selector.observe(on_month_change, names='value')
display(widgets.VBox([month_selector, output]))
on_month_change(None)

In [ ]:
import requests
import datetime
import os
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML
import numpy as np

# --- GitHub Auth ---
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'
if 'GITHUB_TOKEN' not in os.environ:
    raise EnvironmentError("GITHUB_TOKEN not found. Please set it.")

GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}

GRAPHQL_URL = 'https://api.github.com/graphql'

# --- Generate last 6 full months (excluding current) ---
def get_last_6_months():
    today = datetime.date.today()
    first_this_month = datetime.date(today.year, today.month, 1)
    months = []
    for i in range(1, 7):
        month = first_this_month - datetime.timedelta(days=1)
        first_day = datetime.date(month.year, month.month, 1)
        if month.month == 12:
            next_month = datetime.date(month.year + 1, 1, 1)
        else:
            next_month = datetime.date(month.year, month.month + 1, 1)
        last_day = next_month - datetime.timedelta(days=1)
        months.append((first_day.strftime('%B %Y'), first_day.isoformat(), last_day.isoformat()))
        first_this_month = first_day
    return months[::-1]

MONTH_RANGES = get_last_6_months()

# --- Repo Retrieval ---
def get_top_repos(topic = 'javascript', stars = 90000):
    repos = []
    url = f'https://api.github.com/search/repositories?q=topic:{topic}+stars:>={stars}+is:public&per_page=5'
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    items = res.json()['items']
    for repo in items:
        repos.append((repo['owner']['login'], repo['name']))
    return repos

# --- GraphQL Query Template ---
QUERY = '''
query($owner:String!, $name:String!, $since:GitTimestamp!, $until:GitTimestamp!) {
  repository(owner: $owner, name: $name) {
    defaultBranchRef {
      target {
        ... on Commit {
          history(since: $since, until: $until, first: 100) {
            nodes {
              committedDate
              author {
                user {
                  login
                }
              }
            }
            totalCount
          }
        }
      }
    }
  }
}'''

# --- Fetch Contributor Commits ---
def fetch_contributor_commits(owner, repo, since, until):
    variables = {
        'owner': owner,
        'name': repo,
        'since': since + 'T00:00:00Z',
        'until': until + 'T23:59:59Z'
    }
    res = requests.post(GRAPHQL_URL, json={'query': QUERY, 'variables': variables}, headers=HEADERS)
    res.raise_for_status()
    data = res.json()
    nodes = data.get('data', {}).get('repository', {}).get('defaultBranchRef', {}).get('target', {}).get('history', {}).get('nodes', [])
    counts = Counter()
    for node in nodes:
        login = node.get('author', {}).get('user', {}).get('login')
        if login:
            counts[login] += 1
    return counts

# --- Aggregate Median Commits Per Contributor ---
def median_commits_for_month(since, until):
    repos = get_top_repos()
    contributor_counts = Counter()
    for owner, repo in repos:
        try:
            counts = fetch_contributor_commits(owner, repo, since, until)
            contributor_counts.update(counts)
        except:
            continue
    if contributor_counts:
        return int(np.median(list(contributor_counts.values())))
    return 0

# --- Calculate MoM ---
def calculate_mom(current, previous):
    if previous == 0:
        return 'N/A', ''
    pct = round(((current - previous) / previous) * 100, 1)
    arrow = '▲' if pct >= 0 else '▼'
    color = 'green' if pct >= 0 else 'red'
    return f"<span style='color:{color};'>{arrow} {abs(pct)}%</span>", color

# --- KPI Box Renderer ---
def render_kpi_box(current, mom_html, month_label):
    html = f'''
    <div style="border:1px solid #ccc;padding:20px;border-radius:10px;width:300px;background:#f9f9f9">
        <h4 style="margin:0 0 10px 0">Median commits per contributor<br><small style='color:#888;'>in {month_label}</small></h4>
        <div style="font-size:32px;font-weight:bold;">{current}</div>
        <div style="font-size:14px;color:#888;margin-top:5px;">MoM: {mom_html}</div>
    </div>'''
    display(HTML(html))

# --- UI ---
month_selector = widgets.Dropdown(options=[label for label, _, _ in MONTH_RANGES], value=MONTH_RANGES[-1][0])
output = widgets.Output()

def on_month_change(change):
    output.clear_output()
    with output:
        current_idx = [i for i, (label, _, _) in enumerate(MONTH_RANGES) if label == month_selector.value][0]
        if current_idx == 0:
            print("No prior month available to compare.")
            return
        label, since, until = MONTH_RANGES[current_idx]
        _, prev_since, prev_until = MONTH_RANGES[current_idx - 1]
        print(f"Fetching median commit data for {label}...")
        current_val = median_commits_for_month(since, until)
        prev_val = median_commits_for_month(prev_since, prev_until)
        mom_html, _ = calculate_mom(current_val, prev_val)
        render_kpi_box(current_val, mom_html, label)

month_selector.observe(on_month_change, names='value')
display(widgets.VBox([month_selector, output]))
on_month_change(None)

In [ ]:
import requests
import datetime
import os
from collections import defaultdict
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

# --- GitHub Auth ---
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'  # Replace with your actual token
if 'GITHUB_TOKEN' not in os.environ:
    raise EnvironmentError("GITHUB_TOKEN not found. Please set it.")

GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}

# GraphQL Endpoint
GRAPHQL_URL = 'https://api.github.com/graphql'

# Generate last 6 full month ranges (excluding current month)
def get_last_6_months():
    today = datetime.date.today()
    first_this_month = datetime.date(today.year, today.month, 1)
    months = []
    for i in range(1, 7):
        month = first_this_month - datetime.timedelta(days=1)
        first_day = datetime.date(month.year, month.month, 1)
        if month.month == 12:
            next_month = datetime.date(month.year + 1, 1, 1)
        else:
            next_month = datetime.date(month.year, month.month + 1, 1)
        last_day = next_month - datetime.timedelta(days=1)
        months.append((first_day.strftime('%B %Y'), first_day.isoformat(), last_day.isoformat()))
        first_this_month = first_day
    return months[::-1]  # latest first

MONTH_RANGES = get_last_6_months()

# Get top repos with topic 'javascript' and stars >= 100000
def get_top_repos(topic='javascript', stars=100000):
    repos = []
    url = f'https://api.github.com/search/repositories?q=topic:{topic}+stars:>={str(stars)}+is:public&per_page=5'
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    items = res.json()['items']
    for repo in items:
        repos.append((repo['owner']['login'], repo['name']))
    return repos

# GraphQL query template for commits
QUERY = '''
query($owner:String!, $name:String!, $since:GitTimestamp!, $until:GitTimestamp!) {
  repository(owner: $owner, name: $name) {
    defaultBranchRef {
      target {
        ... on Commit {
          history(first: 100, since: $since, until: $until) {
            edges {
              node {
                committedDate
              }
            }
          }
        }
      }
    }
  }
}'''

# Run GraphQL query to get commit dates
def get_commit_dates(owner, repo, since, until):
    variables = {'owner': owner, 'name': repo, 'since': since + 'T00:00:00Z', 'until': until + 'T23:59:59Z'}
    res = requests.post(GRAPHQL_URL, json={'query': QUERY, 'variables': variables}, headers=HEADERS)
    res.raise_for_status()
    data = res.json()
    try:
        commits = data['data']['repository']['defaultBranchRef']['target']['history']['edges']
        return [c['node']['committedDate'][:10] for c in commits]
    except:
        return []

# Count unique active commit days
def get_active_days(since, until):
    repos = get_top_repos()
    all_days = set()
    for owner, repo in repos:
        dates = get_commit_dates(owner, repo, since, until)
        all_days.update(dates)
    return len(all_days)

# Calculate MoM % change
def calc_mom(curr, prev):
    if prev == 0:
        return 'N/A', 'black', ''
    delta = ((curr - prev) / prev) * 100
    arrow = '▲' if delta > 0 else '▼'
    color = 'green' if delta > 0 else 'red'
    return f'{abs(delta):.1f}%', color, arrow

# Render KPI Box
def render_kpi_box(title, value, month_label, change_str, color, arrow):
    html = f'''
    <div style="border:1px solid #ccc;padding:20px;border-radius:10px;width:300px;background:#f9f9f9">
        <h4 style="margin:0 0 10px 0">{title}<br><small style='color:#888;'>({month_label})</small></h4>
        <div style="font-size:32px;font-weight:bold;">{value}</div>
        <div style="color:{color};font-size:18px;font-weight:bold;">MoM: {arrow} {change_str}</div>
    </div>'''
    display(HTML(html))

# Handle month change
month_selector = widgets.Dropdown(options=[label for label, _, _ in MONTH_RANGES], value=MONTH_RANGES[-1][0])
output = widgets.Output()

def on_month_change(change):
    output.clear_output()
    with output:
        idx = next(i for i, (label, _, _) in enumerate(MONTH_RANGES) if label == month_selector.value)
        label, since, until = MONTH_RANGES[idx]
        print(f"Fetching active days for: {label}")
        curr_days = get_active_days(since, until)
        if idx > 0:
            _, prev_since, prev_until = MONTH_RANGES[idx - 1]
            prev_days = get_active_days(prev_since, prev_until)
        else:
            prev_days = 0
        change_str, color, arrow = calc_mom(curr_days, prev_days)
        render_kpi_box("Active commit days", curr_days, label, change_str, color, arrow)

month_selector.observe(on_month_change, names='value')
display(widgets.VBox([month_selector, output]))
on_month_change(None)


In [ ]:
import requests
import datetime
import os
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- GitHub Auth ---
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'  # Replace with your real token
GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}
GRAPHQL_URL = 'https://api.github.com/graphql'

# --- Parameters ---
TOPIC = 'javascript'
MIN_STARS = 100000

# --- Month Ranges ---
def get_last_6_months():
    today = datetime.date.today()
    first_this_month = datetime.date(today.year, today.month, 1)
    months = []
    for _ in range(6):
        last_day = first_this_month - datetime.timedelta(days=1)
        first_day = datetime.date(last_day.year, last_day.month, 1)
        label = first_day.strftime('%B %Y')
        months.append((label, first_day.isoformat(), last_day.isoformat()))
        first_this_month = first_day
    return months[::-1]

MONTH_RANGES = get_last_6_months()

# --- Repo Fetch ---
def get_top_repos():
    url = f'https://api.github.com/search/repositories?q=topic:{TOPIC}+stars:>={MIN_STARS}+is:public&per_page=10'
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    return [(item['owner']['login'], item['name']) for item in res.json()['items']]

# --- GraphQL Query ---
QUERY = '''
query($owner:String!, $name:String!, $since:GitTimestamp!, $until:GitTimestamp!) {
  repository(owner: $owner, name: $name) {
    defaultBranchRef {
      target {
        ... on Commit {
          history(since: $since, until: $until) {
            totalCount
          }
        }
      }
    }
  }
}'''

def get_commit_count(owner, repo, since, until):
    variables = {
        'owner': owner,
        'name': repo,
        'since': since + 'T00:00:00Z',
        'until': until + 'T23:59:59Z'
    }
    res = requests.post(GRAPHQL_URL, json={'query': QUERY, 'variables': variables}, headers=HEADERS)
    res.raise_for_status()
    try:
        return res.json()['data']['repository']['defaultBranchRef']['target']['history']['totalCount']
    except:
        return 0

# --- Chart Function ---
def generate_pie_chart(label, since, until):
    repos = get_top_repos()
    data = {}
    for owner, name in repos:
        count = get_commit_count(owner, name, since, until)
        if count > 0:
            data[f"{owner}/{name}"] = count

    if not data:
        print("No commits found.")
        return

    labels = list(data.keys())
    sizes = list(data.values())

    fig, ax = plt.subplots(figsize=(8, 8))
    wedges, texts, autotexts = ax.pie(
        sizes, labels=None, autopct='%1.1f%%', startangle=140, wedgeprops=dict(width=0.4)
    )
    ax.axis('equal')
    ax.set_title(f"Commits per Repository\nTopic: {TOPIC}, Stars >= {MIN_STARS}, {label}", fontsize=14)
    ax.legend(wedges, labels, title="Repositories", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))
    plt.tight_layout()
    plt.show()

# --- UI ---
month_selector = widgets.Dropdown(
    options=[label for label, _, _ in MONTH_RANGES],
    value=MONTH_RANGES[-1][0],
    description='Month:'
)
run_button = widgets.Button(description='Generate Chart')
output = widgets.Output()

def on_button_click(b):
    output.clear_output()
    with output:
        for label, since, until in MONTH_RANGES:
            if label == month_selector.value:
                print(f"Generating chart for {label}...")
                generate_pie_chart(label, since, until)
                break

run_button.on_click(on_button_click)
display(widgets.VBox([month_selector, run_button, output]))
on_button_click(None)

In [ ]:
import requests
import datetime
import os
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- GitHub Auth ---
os.environ['GITHUB_TOKEN'] = 'ghp_xxx'  # replace with your token
if 'GITHUB_TOKEN' not in os.environ:
    raise EnvironmentError("GITHUB_TOKEN not found. Please set it.")

GITHUB_TOKEN = os.environ['GITHUB_TOKEN']
HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}

GRAPHQL_URL = 'https://api.github.com/graphql'

# --- Month Selector ---
def get_last_6_months():
    today = datetime.date.today()
    first_this_month = datetime.date(today.year, today.month, 1)
    months = []
    for i in range(1, 7):
        month = first_this_month - datetime.timedelta(days=1)
        first_day = datetime.date(month.year, month.month, 1)
        if month.month == 12:
            last_day = datetime.date(month.year + 1, 1, 1) - datetime.timedelta(days=1)
        else:
            last_day = datetime.date(month.year, month.month + 1, 1) - datetime.timedelta(days=1)
        months.append((first_day.strftime('%B %Y'), first_day.isoformat(), last_day.isoformat()))
        first_this_month = first_day
    return months[::-1]

MONTH_RANGES = get_last_6_months()

# --- Top Repos ---
def get_top_repos(topic='javascript', stars=100000):
    repos = []
    url = f'https://api.github.com/search/repositories?q=topic:{topic}+stars:>={stars}+is:public&per_page=5'
    res = requests.get(url, headers=HEADERS)
    res.raise_for_status()
    items = res.json().get('items', [])
    for repo in items:
        repos.append((repo['owner']['login'], repo['name']))
    return repos

# --- GraphQL Commit Query with Pagination ---
QUERY = """
query($owner: String!, $name: String!, $since: GitTimestamp!, $until: GitTimestamp!, $cursor: String) {
  repository(owner: $owner, name: $name) {
    defaultBranchRef {
      target {
        ... on Commit {
          history(since: $since, until: $until, after: $cursor, first: 100) {
            pageInfo {
              hasNextPage
              endCursor
            }
            edges {
              node {
                committedDate
              }
            }
          }
        }
      }
    }
  }
}
"""

def get_commit_dates(owner, name, since, until):
    dates = []
    cursor = None
    while True:
        variables = {
            'owner': owner,
            'name': name,
            'since': since + 'T00:00:00Z',
            'until': until + 'T23:59:59Z',
            'cursor': cursor
        }
        res = requests.post(GRAPHQL_URL, json={'query': QUERY, 'variables': variables}, headers=HEADERS)
        res.raise_for_status()
        data = res.json()
        try:
            history = data['data']['repository']['defaultBranchRef']['target']['history']
        except:
            break
        for edge in history['edges']:
            dates.append(edge['node']['committedDate'][:10])  # 'YYYY-MM-DD'
        if history['pageInfo']['hasNextPage']:
            cursor = history['pageInfo']['endCursor']
        else:
            break
    return dates

# --- Plotting ---
def generate_daily_commit_chart(since, until, label):
    print(f"Generating daily commits chart for {label}...")
    daily_counter = Counter()
    repos = get_top_repos()
    for owner, name in repos:
        print(f"Fetching commits for: {owner}/{name}")
        try:
            dates = get_commit_dates(owner, name, since, until)
            daily_counter.update(dates)
        except Exception as e:
            print(f"Error fetching data for {owner}/{name}: {e}")

    # build date range for selected month
    start_date = datetime.date.fromisoformat(since)
    end_date = datetime.date.fromisoformat(until)
    full_range = pd.date_range(start=start_date, end=end_date)
    daily_counts = [daily_counter.get(date.strftime('%Y-%m-%d'), 0) for date in full_range]

    # plot
    plt.figure(figsize=(14, 5))
    plt.bar(full_range.strftime('%Y-%m-%d'), daily_counts)
    plt.xticks(rotation=90)
    plt.xlabel("Date")
    plt.ylabel("Commits")
    plt.title(f"Daily Commits in {label} (Topic: javascript, Stars >= 100000)")
    plt.tight_layout()
    plt.show()

# --- Widget UI ---
month_selector = widgets.Dropdown(options=[label for label, _, _ in MONTH_RANGES], value=MONTH_RANGES[-1][0])
output = widgets.Output()

def on_month_change(change):
    output.clear_output()
    with output:
        for label, since, until in MONTH_RANGES:
            if label == month_selector.value:
                generate_daily_commit_chart(since, until, label)
                break

month_selector.observe(on_month_change, names='value')
display(widgets.VBox([month_selector, output]))
on_month_change(None)